# Lecture 3 · Kernel Support Vector Machines — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the material of **Lecture 3**: why the SVM dual only needs inner products, the **kernel trick**
$K(\mathbf{x},\mathbf{x}') = \boldsymbol\phi(\mathbf{x})\cdot\boldsymbol\phi(\mathbf{x}')$, valid kernels (Mercer, PSD Gram matrices, composition rules),
the **soft margin** with slack variables $\xi_n$ and the box constraint $0\le\lambda_n\le C$, and kernels for structured data: cosine similarity for
bag-of-words, the $p$-spectrum kernel and the all-subsequences kernel with its dynamic programme. Code exercises implement kernels and a small dual
solver in NumPy/SciPy and connect them to `SVC` as used in `MLSelfStudy3-F26.ipynb`.

**20 exercises** · 🧠 Concept ×5 · ✍️ By hand ×8 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import itertools
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel, rbf_kernel
from scipy.optimize import minimize
rng = np.random.default_rng(3)

def subsequence_counts(s):
    # brute force: phi^+_u(s) for every u (including the empty string). Exponential: only for short strings!
    c = Counter()
    for k in range(len(s) + 1):
        for idx in itertools.combinations(range(len(s)), k):
            c[''.join(s[i] for i in idx)] += 1
    return c

## Part A · Concepts

### Exercise 1 · What the kernel trick buys
*🧠 Concept · ★☆☆*

(a) Where do the data appear in the SVM dual objective and in the prediction $f(\mathbf{z})$?
(b) Explain in your own words what the kernel trick replaces and why that is cheaper.
(c) Why can the RBF kernel $\exp(-\gamma\lVert\mathbf{x}-\mathbf{x}'\rVert^2)$ not be used with an explicit $\boldsymbol\phi$ at all?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at $\sum_{n,m}\lambda_n\lambda_my_ny_m(\mathbf{x}_n\cdot\mathbf{x}_m)$ and $f(\mathbf{z}) = \sum_{n\in SV}\lambda_ny_n(\mathbf{x}_n\cdot\mathbf{z}) + b$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Only through inner products $\mathbf{x}_n\cdot\mathbf{x}_m$ (training) and $\mathbf{x}_n\cdot\mathbf{z}$ (prediction); $\mathbf{w}$ never needs to be formed.

(b) We replace every $\boldsymbol\phi(\mathbf{x})\cdot\boldsymbol\phi(\mathbf{x}')$ by a function $K(\mathbf{x},\mathbf{x}')$ evaluated in input space. For the degree-$d$ polynomial kernel
that costs $O(D)$ instead of $O\big(\binom{D+d}{d}\big)$ for the explicit feature vectors (Exercise 7).

(c) Its feature space is infinite-dimensional (expand $e^{2\gamma\mathbf{x}\cdot\mathbf{x}'}$ as a power series: all polynomial degrees appear), so $\boldsymbol\phi$ cannot be stored, but
$K$ is a single exponential.

</details>

### Exercise 2 · Reading slack variables and multipliers
*🧠 Concept · ★★☆*

For a trained soft-margin SVM, describe where a point lies (relative to the margin and the boundary) in each case, and whether it is a support vector:

(a) $\xi_n = 0,\ \lambda_n = 0$; (b) $\xi_n = 0,\ 0<\lambda_n<C$; (c) $0<\xi_n\le1,\ \lambda_n = C$; (d) $\xi_n>1,\ \lambda_n = C$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The constraint is $y_n f(\mathbf{x}_n)\ge 1-\xi_n$; complementary slackness links $\lambda_n>0$ to equality.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Correctly classified, on or outside the margin, **not** a support vector (does not influence $\mathbf{w}$).
(b) Exactly on the margin, $y_nf(\mathbf{x}_n) = 1$: a "free" support vector; these are used to compute $b$.
(c) Inside the margin but on the correct side ($0\le y_nf<1$): a bounded support vector.
(d) On the wrong side of the boundary (misclassified): a bounded support vector.

</details>

### Exercise 3 · The two extremes of C
*🧠 Concept · ★★☆*

(a) Reproduce the argument from `exercise3 solutions.pdf`: what is the optimal solution for $C = 0$ and what does the classifier predict?
(b) What happens as $C\to\infty$ on separable data, and on non-separable data?
(c) In terms of bias and variance, which direction of $C$ risks overfitting?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

With $C=0$ the objective is just $\frac12\lVert\mathbf{w}\rVert^2$ and slack is free.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Minimising $\frac12\lVert\mathbf{w}\rVert^2$ gives $\mathbf{w} = \mathbf{0}$; constraints $y_nb\ge1-\xi_n$ are satisfied by large slack for any $b$, so $f(\mathbf{x}) = b$ is constant: every point gets the same label. Useless, so $C>0$ is required.
(b) Large $C$ makes slack expensive: on separable data it approaches the **hard-margin** SVM; on non-separable data it tries hard to classify every point, giving a narrow margin and a wiggly (kernel) boundary.
(c) Large $C$ means low bias and high variance (overfitting); small $C$ means a wide margin, more support vectors and higher bias. Exercise 20 shows the trend.

</details>

### Exercise 4 · Kernel SVMs at scale and with many classes
*🧠 Concept · ★★☆*

(a) How much memory does a dense Gram matrix need for $N = 60\,000$ MNIST images in float64?
(b) For $K = 10$ classes, how many binary SVMs do OvR and OvO train, and roughly how many training points does each see (balanced classes, $N$ total)?
(c) Assuming training costs $\propto N^2$, compare the total cost of OvR and OvO.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$N^2\cdot8$ bytes. OvO: each model sees two classes, i.e. $2N/K$ points.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $60\,000^2\cdot8 \approx 2.9\times10^{10}$ bytes $\approx$ **28.8 GB** (that is why LIBSVM caches only parts and why the self-study uses a GPU / PCA).
(b) OvR: 10 models, each on all $N$ points. OvO: $\binom{10}{2} = 45$ models on $2N/10 = N/5$ points each.
(c) OvR $\approx 10N^2$; OvO $\approx 45(N/5)^2 = 1.8N^2$. OvO is about $5.6\times$ cheaper despite training more models, the point made in the Critical Insights.

</details>

### Exercise 5 · PCA to 2-D, then a polynomial kernel
*🧠 Concept · ★★☆*

The self-study first projects MNIST to 2 principal components and then trains `SVC(kernel="poly", degree=3, C=10)`.
(a) What kind of map is each step (linear/non-linear, supervised/unsupervised)? (b) Why can the accuracy still be poor even with a flexible kernel?
(c) Suggest a better pipeline.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Can a kernel separate points that PCA has already put on top of each other?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) PCA: linear, unsupervised projection maximising variance. Polynomial kernel: an implicit non-linear map; the SVM is supervised.
(b) PCA ignores the labels; digits that overlap in the 2-D projection are **identical** inputs to the SVM, so no kernel can separate them. Information lost by the projection
cannot be recovered.
(c) Keep many more components (e.g. 50 to 100, or all pixels) and let the kernel (e.g. RBF on standardised inputs) do the non-linear work; tune $C$ and $\gamma$ by cross-validation.

</details>

## Part B · By hand

### Exercise 6 · A polynomial kernel evaluation, two ways
*✍️ By hand · ★☆☆*

For $\mathbf{x} = (1, 2)$ and $\mathbf{x}' = (3, -1)$ compute $K(\mathbf{x},\mathbf{x}') = (\mathbf{x}\cdot\mathbf{x}'+1)^2$ directly, and then via the explicit map
$\boldsymbol\phi(\mathbf{x}) = (x_1^2, x_2^2, \sqrt2x_1, \sqrt2x_2, \sqrt2x_1x_2, 1)$. Also compute the RBF value $\exp(-\gamma\lVert\mathbf{x}-\mathbf{x}'\rVert^2)$ for $\gamma = 0.1$.

In [ ]:
k_poly = None
k_rbf = None

_x, _z = np.array([[1, 2.]]), np.array([[3, -1.]])
check('poly kernel', k_poly, polynomial_kernel(_x, _z, degree=2, gamma=1, coef0=1)[0, 0])
check('rbf kernel', k_rbf, rbf_kernel(_x, _z, gamma=0.1)[0, 0], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbf{x}\cdot\mathbf{x}' = 3 - 2 = 1$. $\lVert\mathbf{x}-\mathbf{x}'\rVert^2 = 4 + 9$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Direct: $(1 + 1)^2 = \mathbf{4}$. Via $\boldsymbol\phi$: $\boldsymbol\phi(\mathbf{x}) = (1, 4, \sqrt2, 2\sqrt2, 2\sqrt2, 1)$, $\boldsymbol\phi(\mathbf{x}') = (9, 1, 3\sqrt2, -\sqrt2, -3\sqrt2, 1)$,
inner product $9 + 4 + 6 - 4 - 12 + 1 = 4$. ✔ RBF: $e^{-0.1\cdot13} = e^{-1.3}\approx\mathbf{0.2725}$.

```python
k_poly = 4.0
k_rbf = np.exp(-1.3)
```

</details>

### Exercise 7 · Feature-space dimension of the polynomial kernel
*✍️ By hand · ★★☆*

(a) Show that $K(\mathbf{x},\mathbf{x}') = (\mathbf{x}\cdot\mathbf{x}')^2$ on $\mathbb{R}^2$ corresponds to $\boldsymbol\phi(\mathbf{x}) = (x_1^2, x_2^2, \sqrt2x_1x_2)$.
(b) $(\mathbf{x}\cdot\mathbf{x}'+1)^d$ on $\mathbb{R}^D$ spans all monomials of degree $\le d$, of which there are $\binom{D+d}{d}$. Check this gives 6 for $D=d=2$ and
compute the number for MNIST pixels with the self-study's cubic kernel ($D = 784$, $d = 3$).

In [ ]:
n_monomials_mnist = None

from math import comb
check('number of monomials', n_monomials_mnist, comb(784 + 3, 3))

<details>
<summary><b>💡 Hint</b></summary>

$\binom{787}{3} = \frac{787\cdot786\cdot785}{6}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $(x_1x_1' + x_2x_2')^2 = x_1^2x_1'^2 + x_2^2x_2'^2 + 2x_1x_2x_1'x_2' = \boldsymbol\phi(\mathbf{x})\cdot\boldsymbol\phi(\mathbf{x}')$.

(b) $\binom{4}{2} = 6$ ✔ (the six coordinates in the note). For MNIST: $\binom{787}{3} = \mathbf{80\,931\,145}$ features per image; the kernel needs one 784-dim dot product and a cube.

```python
n_monomials_mnist = 787 * 786 * 785 // 6
```

</details>

### Exercise 8 · Is it a kernel? Check the Gram matrix
*✍️ By hand · ★★☆*

Someone proposes the "similarity" $K(x, x') = (x - x')^2$ on $\mathbb{R}$. Build its Gram matrix for the two points $x_1 = 0$, $x_2 = 1$, find its eigenvalues and decide
whether $K$ is a valid kernel. Then use the composition rules to argue that $K(\mathbf{x},\mathbf{x}') = \exp(\mathbf{x}\cdot\mathbf{x}')\,(\mathbf{x}\cdot\mathbf{x}'+1)^2$ **is** valid.

In [ ]:
min_eig_bad = None

check('smallest eigenvalue', min_eig_bad, np.linalg.eigvalsh(np.array([[0, 1], [1, 0.]])).min())

<details>
<summary><b>💡 Hint</b></summary>

A valid kernel must have $K(x,x)\ge0$ and a PSD Gram matrix. What is the trace of this Gram matrix?

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbf{K} = \begin{pmatrix}0&1\\1&0\end{pmatrix}$ with eigenvalues $\pm1$; the smallest is $\mathbf{-1} < 0$ (e.g. $\mathbf{v} = (1,-1)$: $\mathbf{v}^T\mathbf{K}\mathbf{v} = -2$),
so it is **not** PSD and not a kernel (it is a squared *distance*, not a similarity).

The second one: $\mathbf{x}\cdot\mathbf{x}'$ is a kernel; $\exp(K_1)$ is a kernel; $q(K_1) = (K_1+1)^2$ is a polynomial with non-negative coefficients, so a kernel;
the product of two kernels is a kernel. ✔

```python
min_eig_bad = -1.0
```

</details>

### Exercise 9 · Cosine kernel for two documents
*✍️ By hand · ★☆☆*

Vocabulary (*ball, goal, game, vote*). Document $t_1$ has counts $(2, 1, 0, 1)$, $t_2$ has $(1, 1, 1, 0)$. Compute $K_{\cos}(t_1,t_2)$.
Why is the normalisation useful for documents of very different lengths? Which composition rule makes it a valid kernel?

In [ ]:
k_cos = None

_a, _b = np.array([2, 1, 0, 1.]), np.array([1, 1, 1, 0.])
from scipy.spatial.distance import cosine
check('cosine kernel', k_cos, 1 - cosine(_a, _b), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbf{tf}(t_1)\cdot\mathbf{tf}(t_2) = 3$, $\lVert\mathbf{tf}(t_1)\rVert = \sqrt6$, $\lVert\mathbf{tf}(t_2)\rVert = \sqrt3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$K_{\cos} = 3/\sqrt{18} = 1/\sqrt2\approx\mathbf{0.7071}$. Without normalisation a document repeated ten times would look ten times "more similar" to everything; cosine
measures only the direction (topic mix). It is the **normalisation** rule $K_1/\sqrt{K_1(\mathbf{x},\mathbf{x})K_1(\mathbf{x}',\mathbf{x}')}$ applied to the linear kernel on $\mathbf{tf}$.

```python
k_cos = 3 / np.sqrt(18)
```

</details>

### Exercise 10 · Substrings vs. subsequences of 'statistics'
*✍️ By hand · ★★☆*

For $\mathbf{s} = $ "statistics", compute by hand $\phi_u(\mathbf{s})$ (contiguous) and $\phi^+_u(\mathbf{s})$ (gaps allowed) for $u = $ "st" and $u = $ "ti".

In [ ]:
phi_st = None       # substring count of 'st'
phi_plus_st = None  # subsequence count of 'st'
phi_ti = None
phi_plus_ti = None

_s = 'statistics'
_sub = lambda u: sum(_s[i:i+len(u)] == u for i in range(len(_s) - len(u) + 1))
_seq = subsequence_counts(_s)
check("phi_st", phi_st, _sub('st')); check("phi+_st", phi_plus_st, _seq['st'])
check("phi_ti", phi_ti, _sub('ti')); check("phi+_ti", phi_plus_ti, _seq['ti'])

<details>
<summary><b>💡 Hint</b></summary>

Positions: s$_1$ t$_2$ a$_3$ t$_4$ i$_5$ s$_6$ t$_7$ i$_8$ c$_9$ s$_{10}$. For subsequences count pairs (position of first letter < position of second).

</details>

<details>
<summary><b>✅ Solution</b></summary>

"st": contiguous at positions (1,2) and (6,7), so $\phi = \mathbf{2}$. Subsequences: s$_1$ with t$_2$, t$_4$, t$_7$ (3) plus s$_6$ with t$_7$ (1): $\phi^+ = \mathbf{4}$.

"ti": contiguous at (4,5) and (7,8): $\phi = \mathbf{2}$. Subsequences: t$_2$ with i$_5$, i$_8$ (2), t$_4$ with i$_5$, i$_8$ (2), t$_7$ with i$_8$ (1): $\phi^+ = \mathbf{5}$ (as in the note's table).

```python
phi_st, phi_plus_st, phi_ti, phi_plus_ti = 2, 4, 2, 5
```

</details>

### Exercise 11 · A 2-spectrum kernel by hand
*✍️ By hand · ★☆☆*

Compute $K_2(\mathbf{s},\mathbf{t}) = \sum_{u\in\Sigma^2}\phi_u(\mathbf{s})\phi_u(\mathbf{t})$ for $\mathbf{s} = $ "abab" and $\mathbf{t} = $ "bab", and the normalised version
$K_2(\mathbf{s},\mathbf{t})/\sqrt{K_2(\mathbf{s},\mathbf{s})K_2(\mathbf{t},\mathbf{t})}$.

In [ ]:
k2_st = None
k2_norm = None

_c = lambda s: Counter(s[i:i+2] for i in range(len(s) - 1))
_k = lambda a, b: sum(_c(a)[u] * _c(b)[u] for u in _c(a))
check('K_2(s,t)', k2_st, _k('abab', 'bab'))
check('normalised', k2_norm, _k('abab', 'bab') / np.sqrt(_k('abab', 'abab') * _k('bab', 'bab')), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

2-grams of 'abab': ab, ba, ab. Of 'bab': ba, ab.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\boldsymbol\phi(\text{abab})$: ab$\to2$, ba$\to1$; $\boldsymbol\phi(\text{bab})$: ab$\to1$, ba$\to1$. $K_2 = 2\cdot1 + 1\cdot1 = \mathbf{3}$.
$K_2(\mathbf{s},\mathbf{s}) = 4 + 1 = 5$, $K_2(\mathbf{t},\mathbf{t}) = 2$, so the normalised value is $3/\sqrt{10}\approx\mathbf{0.949}$.

```python
k2_st = 3
k2_norm = 3 / np.sqrt(10)
```

</details>

### Exercise 12 · Predicting with a kernel SVM by hand
*✍️ By hand · ★★☆*

A trained SVM with kernel $K(\mathbf{x},\mathbf{z}) = (\mathbf{x}\cdot\mathbf{z}+1)^2$ has two support vectors $\mathbf{x}_1 = (0,0)$, $y_1 = +1$, $\lambda_1 = 1$ and
$\mathbf{x}_2 = (1,1)$, $y_2 = -1$, $\lambda_2 = 1$, and $b = 1$. Compute $f(\mathbf{z}) = \sum_n\lambda_ny_nK(\mathbf{x}_n,\mathbf{z}) + b$ for $\mathbf{z}_a = (1,0)$ and
$\mathbf{z}_b = (-1,0)$, and give the predicted labels. (Check that $\sum_n\lambda_ny_n = 0$ holds.)

In [ ]:
f_za = None
f_zb = None

_SV = np.array([[0, 0], [1, 1.]]); _coef = np.array([1, -1.])
_f = lambda z: _coef @ polynomial_kernel(_SV, np.array([z], float), degree=2, gamma=1, coef0=1)[:, 0] + 1
check('f(z_a)', f_za, _f([1, 0])); check('f(z_b)', f_zb, _f([-1, 0]))

<details>
<summary><b>💡 Hint</b></summary>

$K(\mathbf{x}_1,\cdot) = 1$ for every $\mathbf{z}$ because $\mathbf{x}_1 = \mathbf{0}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sum\lambda_ny_n = 1 - 1 = 0$ ✔.
$\mathbf{z}_a$: $K_1 = 1$, $K_2 = (1+1)^2 = 4$, so $f = 1 - 4 + 1 = \mathbf{-2}$, label $-1$.
$\mathbf{z}_b$: $K_1 = 1$, $K_2 = (-1+1)^2 = 0$, so $f = 1 - 0 + 1 = \mathbf{2}$, label $+1$.

```python
f_za = -2.0
f_zb = 2.0
```

</details>

### Exercise 13 · Where does the box constraint come from?
*✍️ By hand · ★★★*

Write the Lagrangian of the soft-margin primal $\min\frac12\lVert\mathbf{w}\rVert^2 + C\sum_n\xi_n$ s.t. $y_n(\mathbf{w}\cdot\mathbf{x}_n+b)\ge1-\xi_n$, $\xi_n\ge0$,
using multipliers $\lambda_n\ge0$ and $\mu_n\ge0$. Derive the dual and show that the only change from the hard margin is $0\le\lambda_n\le C$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Differentiate w.r.t. $\xi_n$: $C - \lambda_n - \mu_n = 0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$L = \frac12\lVert\mathbf{w}\rVert^2 + C\sum\xi_n - \sum\lambda_n\{y_n(\mathbf{w}\cdot\mathbf{x}_n+b) - 1 + \xi_n\} - \sum\mu_n\xi_n$.

$\partial_\mathbf{w}$: $\mathbf{w} = \sum\lambda_ny_n\mathbf{x}_n$; $\partial_b$: $\sum\lambda_ny_n = 0$; $\partial_{\xi_n}$: $C - \lambda_n - \mu_n = 0$.
Substituting, all $\xi_n$ terms cancel ($C\xi_n - \lambda_n\xi_n - \mu_n\xi_n = 0$), leaving the **same** dual objective
$\sum\lambda_n - \frac12\sum_{n,m}\lambda_n\lambda_my_ny_m\mathbf{x}_n\cdot\mathbf{x}_m$. Since $\mu_n = C - \lambda_n\ge0$, we get $0\le\lambda_n\le C$.
Complementary slackness $\mu_n\xi_n = 0$: if $\lambda_n<C$ then $\mu_n>0$ so $\xi_n = 0$; points with $\xi_n>0$ must have $\lambda_n = C$ (Exercise 2).

</details>

## Part C · Code from scratch

### Exercise 14 · Gram matrices in NumPy
*💻 Code · ★☆☆*

Implement vectorised `gram_linear(A, B)`, `gram_poly(A, B, d, c=1)` $=(\mathbf{a}\cdot\mathbf{b}+c)^d$ and `gram_rbf(A, B, gamma)` returning the $|A|\times|B|$ kernel matrices,
without Python loops. Compare with `sklearn.metrics.pairwise`.

In [ ]:
A = rng.normal(size=(6, 3)); B = rng.normal(size=(4, 3))

def gram_linear(A, B):
    return None
def gram_poly(A, B, d, c=1.0):
    return None
def gram_rbf(A, B, gamma):
    # hint: ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a.b
    return None

check('linear', gram_linear(A, B), linear_kernel(A, B))
check('poly d=3', gram_poly(A, B, 3), polynomial_kernel(A, B, degree=3, gamma=1, coef0=1))
check('rbf', gram_rbf(A, B, 0.5), rbf_kernel(A, B, gamma=0.5))

<details>
<summary><b>💡 Hint</b></summary>

`(A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T` gives all squared distances.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Broadcasting turns the double loop into matrix products; clip tiny negative squared distances to 0 to be safe.

```python
A = rng.normal(size=(6, 3)); B = rng.normal(size=(4, 3))

def gram_linear(A, B):
    return A @ B.T
def gram_poly(A, B, d, c=1.0):
    return (A @ B.T + c) ** d
def gram_rbf(A, B, gamma):
    sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.exp(-gamma * np.maximum(sq, 0))
```

</details>

### Exercise 15 · Explicit features vs. kernel, and PSD checks
*💻 Code · ★★☆*

1. Implement `phi6(X)`, the explicit 6-D map of the note, and verify `phi6(X) @ phi6(X).T` equals `gram_poly(X, X, 2)` on 20 random 2-D points (`max_diff_phi`).
2. Compute the smallest eigenvalue of the Gram matrix on the same points for: linear, poly($d=3$), RBF($\gamma=1$), and the non-kernel $(\mathbf{x}-\mathbf{x}')$ squared distance
   $\lVert\mathbf{x}-\mathbf{x}'\rVert^2$. Store them in the dict `min_eigs` with keys `'linear', 'poly', 'rbf', 'sqdist'`.

In [ ]:
X20 = rng.normal(size=(20, 2))

def phi6(X):
    # TODO: columns x1^2, x2^2, sqrt2 x1, sqrt2 x2, sqrt2 x1 x2, 1
    return None

max_diff_phi = None
min_eigs = None

check('phi6 reproduces the kernel', None if max_diff_phi is None else max_diff_phi < 1e-9, True)
if min_eigs is not None:
    check('valid kernels are PSD', min(min_eigs['linear'], min_eigs['poly'], min_eigs['rbf']) > -1e-8, True)
    _D = ((X20[:, None] - X20[None]) ** 2).sum(-1)
    check('squared distance is not', min_eigs['sqdist'], np.linalg.eigvalsh(_D).min())
else:
    check('valid kernels are PSD', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.eigvalsh` for symmetric matrices. Tiny negative values like $-10^{-13}$ are round-off.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The explicit map reproduces the kernel to machine precision. The three kernels have eigenvalues $\ge0$ (up to round-off), while the squared-distance matrix has zero
trace and so necessarily a clearly negative eigenvalue: it violates the Gram-matrix criterion.

```python
X20 = rng.normal(size=(20, 2))

def phi6(X):
    x1, x2 = X[:, 0], X[:, 1]
    s = np.sqrt(2)
    return np.c_[x1 ** 2, x2 ** 2, s * x1, s * x2, s * x1 * x2, np.ones(len(X))]

max_diff_phi = np.abs(phi6(X20) @ phi6(X20).T - gram_poly(X20, X20, 2)).max()
D2 = ((X20[:, None] - X20[None]) ** 2).sum(-1)
min_eigs = {k: np.linalg.eigvalsh(G).min() for k, G in
            [('linear', gram_linear(X20, X20)), ('poly', gram_poly(X20, X20, 3)), ('rbf', gram_rbf(X20, X20, 1.0)), ('sqdist', D2)]}
print({k: round(v, 4) for k, v in min_eigs.items()})
```

</details>

### Exercise 16 · The p-spectrum kernel
*💻 Code · ★★☆*

Implement `spectrum_kernel(s, t, p)` with a `Counter` of the length-$p$ substrings, and `spectrum_gram(strings, p, normalise)` (optionally with the
normalisation rule). Compute the normalised 3-spectrum Gram matrix for the four DNA strings below and store it in `G_spec`.
Which pair is most similar?

In [ ]:
dna = ['GATTACA', 'GATTACCA', 'TACAGAT', 'CCCGGG']

def spectrum_kernel(s, t, p):
    return None

def spectrum_gram(strings, p, normalise=True):
    return None

G_spec = spectrum_gram(dna, 3)

def _phi(s, p):   # explicit feature vector over all 4^p possible p-mers
    return np.array([sum(s[i:i+p] == ''.join(u) for i in range(len(s) - p + 1)) for u in itertools.product('ACGT', repeat=p)], float)
_F = np.array([_phi(s, 3) for s in dna]); _G = _F @ _F.T
check('normalised 3-spectrum Gram', G_spec, _G / np.sqrt(np.outer(np.diag(_G), np.diag(_G))))

<details>
<summary><b>💡 Hint</b></summary>

`Counter(s[i:i+p] for i in range(len(s)-p+1))`; only p-mers present in both strings contribute.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Iterating only over the p-mers that occur is $O(|s|+|t|)$, compared with the $|\Sigma|^p$-dimensional explicit vector the check builds.
GATTACA and GATTACCA share GAT, ATT, TTA, TAC and are the most similar pair; CCCGGG shares nothing with the others (similarity 0).

```python
dna = ['GATTACA', 'GATTACCA', 'TACAGAT', 'CCCGGG']

def spectrum_kernel(s, t, p):
    cs = Counter(s[i:i+p] for i in range(len(s) - p + 1))
    ct = Counter(t[i:i+p] for i in range(len(t) - p + 1))
    return sum(cs[u] * ct[u] for u in cs)

def spectrum_gram(strings, p, normalise=True):
    G = np.array([[spectrum_kernel(a, b, p) for b in strings] for a in strings], float)
    if normalise:
        d = np.sqrt(np.diag(G))
        G = G / np.outer(d, d)
    return G

G_spec = spectrum_gram(dna, 3)
print(G_spec.round(3))
```

</details>

### Exercise 17 · All-subsequences kernel by dynamic programming
*💻 Code · ★★★*

Implement the recursion of the note,
$$K(\mathbf{s}[1{:}i], \mathbf{t}[1{:}j]) = K(\mathbf{s}[1{:}i-1], \mathbf{t}[1{:}j]) + \sum_{k\le j:\ t_k = s_i}K(\mathbf{s}[1{:}i-1], \mathbf{t}[1{:}k-1]),$$
with $K(\epsilon,\mathbf{t}) = K(\mathbf{s}, \epsilon) = 1$, as a table `K[i, j]` of shape $(|s|+1)\times(|t|+1)$. Evaluate it on the pairs in `pairs` and store the values in `k_all`.
(The check compares with brute-force enumeration of all subsequences.)

In [ ]:
pairs = [('gatta', 'cata'), ('abab', 'bab'), ('statistics', 'stats')]

def all_subseq_kernel(s, t):
    # TODO: DP table
    return None

k_all = [all_subseq_kernel(s, t) for s, t in pairs]

_bf = []
for s, t in pairs:
    a, b = subsequence_counts(s), subsequence_counts(t)
    _bf.append(sum(a[u] * b[u] for u in a))
check('all-subsequences kernel', None if k_all[0] is None else k_all, _bf)

<details>
<summary><b>💡 Hint 1</b></summary>

Row 0 and column 0 of the table are all ones.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

The inner sum makes this $O(|s|\,|t|^2)$; keeping a running partial sum per row brings it down to $O(|s|\,|t|)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The explicit feature space ($\Sigma^*$) is infinite and the brute-force check is exponential in the string length, but the DP fills an $(n+1)\times(m+1)$ table.
For "abab" vs "bab" the value 13 counts every common subsequence pair, including the empty string.

```python
pairs = [('gatta', 'cata'), ('abab', 'bab'), ('statistics', 'stats')]

def all_subseq_kernel(s, t):
    n, m = len(s), len(t)
    K = np.ones((n + 1, m + 1))
    for i in range(1, n + 1):
        running = 0.0                          # sum_{k<=j, t_k = s_i} K[i-1, k-1]
        for j in range(1, m + 1):
            if t[j - 1] == s[i - 1]:
                running += K[i - 1, j - 1]
            K[i, j] = K[i - 1, j] + running
    return K[n, m]

k_all = [all_subseq_kernel(s, t) for s, t in pairs]
print(k_all)
```

</details>

### Exercise 18 · An RBF decision function from the dual coefficients
*💻 Code · ★★☆*

Fit `SVC(kernel='rbf', gamma=1.0, C=1.0)` on `make_moons`. Using only `support_vectors_`, `dual_coef_` ($=\lambda_ny_n$), `intercept_` and your `gram_rbf`,
compute $f(\mathbf{z}) = \sum_{n\in SV}\lambda_ny_nK(\mathbf{x}_n,\mathbf{z}) + b$ for 50 test points (`f_manual`). Also check the box constraint: store $\max_n\lambda_n$ in `max_lambda`.

In [ ]:
X_m, y_m = make_moons(200, noise=0.25, random_state=0)
Z_m = rng.uniform(-1.5, 2.5, size=(50, 2))
svm_rbf = SVC(kernel='rbf', gamma=1.0, C=1.0).fit(X_m, y_m)
f_manual = None
max_lambda = None

check('manual decision function', f_manual, svm_rbf.decision_function(Z_m))
check('lambda_n <= C', None if max_lambda is None else max_lambda <= 1.0 + 1e-9, True)

<details>
<summary><b>💡 Hint</b></summary>

`svm_rbf.dual_coef_` has shape (1, n_SV); multiply by the (n_SV, 50) kernel matrix.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Prediction costs one kernel evaluation per support vector, so fewer SVs means faster predictions. Many multipliers sit exactly at $C$ (bounded SVs inside the margin),
which is the $0\le\lambda_n\le C$ box from Exercise 13.

```python
X_m, y_m = make_moons(200, noise=0.25, random_state=0)
Z_m = rng.uniform(-1.5, 2.5, size=(50, 2))
svm_rbf = SVC(kernel='rbf', gamma=1.0, C=1.0).fit(X_m, y_m)
f_manual = svm_rbf.dual_coef_[0] @ gram_rbf(svm_rbf.support_vectors_, Z_m, 1.0) + svm_rbf.intercept_[0]
max_lambda = np.abs(svm_rbf.dual_coef_).max()
print('SVs:', len(svm_rbf.support_), ' at the bound C:', np.sum(np.isclose(np.abs(svm_rbf.dual_coef_), 1.0)))
```

</details>

### Exercise 19 · Solve the soft-margin dual yourself
*💻 Code · ★★★*

Solve the dual $\max_\boldsymbol\lambda\sum\lambda_n - \frac12\boldsymbol\lambda^TQ\boldsymbol\lambda$ with $Q_{nm} = y_ny_mK_{nm}$, subject to $0\le\lambda_n\le C$ and
$\sum\lambda_ny_n = 0$, using `scipy.optimize.minimize(method='SLSQP')` (minimise the negative). Linear kernel, $C = 1$, data below.
Recover $\mathbf{w} = \sum\lambda_ny_n\mathbf{x}_n$ and $b$ as the average of $y_n - \mathbf{w}\cdot\mathbf{x}_n$ over the **free** SVs ($0<\lambda_n<C$).
Store `w_qp`, `b_qp` and compare with `SVC(kernel='linear', C=1)`.

In [ ]:
r = np.random.default_rng(4)
X_qp = np.r_[r.normal([-1, -1], 0.9, (15, 2)), r.normal([1, 1], 0.9, (15, 2))]
y_qp = np.r_[-np.ones(15), np.ones(15)]
C_qp = 1.0
w_qp = None
b_qp = None

_s = SVC(kernel='linear', C=1.0).fit(X_qp, y_qp)
check('w', w_qp, _s.coef_[0], tol=1e-3); check('b', b_qp, _s.intercept_[0], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Pass `bounds=[(0, C)]*N`, `constraints=[{'type': 'eq', 'fun': lambda l: l @ y}]`, and the analytic gradient `jac=lambda l: Q @ l - 1`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use a tight tolerance: `options={'ftol': 1e-12, 'maxiter': 500}`. Free SVs: `(lam > 1e-6) & (lam < C - 1e-6)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SLSQP solves this small quadratic programme in a fraction of a second and matches LIBSVM to about $10^{-4}$. LIBSVM uses SMO (optimising two multipliers at a time),
which scales to far more points than a generic QP solver, but solves the same problem.

```python
r = np.random.default_rng(4)
X_qp = np.r_[r.normal([-1, -1], 0.9, (15, 2)), r.normal([1, 1], 0.9, (15, 2))]
y_qp = np.r_[-np.ones(15), np.ones(15)]
C_qp = 1.0
N = len(y_qp)
Q = (y_qp[:, None] * y_qp[None, :]) * gram_linear(X_qp, X_qp)
res = minimize(lambda l: 0.5 * l @ Q @ l - l.sum(), np.zeros(N), jac=lambda l: Q @ l - 1,
               method='SLSQP', bounds=[(0, C_qp)] * N,
               constraints=[{'type': 'eq', 'fun': lambda l: l @ y_qp, 'jac': lambda l: y_qp}],
               options={'ftol': 1e-12, 'maxiter': 500})
lam = res.x
w_qp = (lam * y_qp) @ X_qp
free = (lam > 1e-6) & (lam < C_qp - 1e-6)
b_qp = np.mean(y_qp[free] - X_qp[free] @ w_qp)
print('w', w_qp.round(4), 'b', round(b_qp, 4), ' SVs:', np.sum(lam > 1e-6), ' free:', free.sum())
```

</details>

### Exercise 20 · C controls the number of support vectors
*💻 Code · ★★☆*

On the moons data (`X_m, y_m`) fit an RBF SVM ($\gamma = 1$) for $C\in\{0.01, 0.1, 1, 10, 100\}$ and record the number of support vectors in the dict `n_sv_by_C`
and the training accuracy in `acc_by_C`. Explain the trend. Optional: plot the decision boundaries.

In [ ]:
Cs = [0.01, 0.1, 1, 10, 100]
n_sv_by_C = None
acc_by_C = None

if n_sv_by_C is not None:
    check('fewer SVs as C grows', all(n_sv_by_C[a] >= n_sv_by_C[b] for a, b in zip(Cs, Cs[1:])), True)
    check('training accuracy grows overall', acc_by_C[100] > acc_by_C[0.01], True)
else:
    check('fewer SVs as C grows', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`model.n_support_.sum()` counts the support vectors.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Small $C$: slack is cheap, the margin is wide and nearly all points are inside it (with $C=0.01$ all 200 are SVs) and the boundary is smooth (underfit).
Large $C$: few margin violations allowed, few SVs, higher training accuracy and a more complex boundary. Choose $C$ (and $\gamma$) by validation, not training accuracy.

```python
Cs = [0.01, 0.1, 1, 10, 100]
n_sv_by_C, acc_by_C = {}, {}
for c in Cs:
    m = SVC(kernel='rbf', gamma=1.0, C=c).fit(X_m, y_m)
    n_sv_by_C[c] = int(m.n_support_.sum())
    acc_by_C[c] = m.score(X_m, y_m)
print(n_sv_by_C); print(acc_by_C)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*